# Forklift Anomaly Detection Skeleton


## 0. パッケージのインストール


In [ ]:
#%pip install jupyterlab==4.4.1 matplotlib==3.10.1 numpy==2.2.4 pandas==2.2.3 scipy==1.15.2 scikit-learn==1.6.1 librosa==0.10.2.post1 soundfile==0.13.1 opencv-python-headless==4.11.0.86


In [ ]:
from __future__ import annotations

from io import BytesIO
from pathlib import Path
from typing import Any

import shutil
import subprocess

import cv2
import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display
from scipy import signal
from scipy.spatial import Delaunay
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler


In [ ]:
DATA_DIR = Path("../data")
NORMAL_VIDEO_DIR = DATA_DIR / "normal"
ANOMARY_VIDEO_DIR = DATA_DIR / "anomary"
REFERENCE_VIDEO_PATH = DATA_DIR / "sample.mp4"
OUTPUT_DIR = Path("../outputs")
ANOMARY_LIKE_NORMAL_LIST_PATH = DATA_DIR / "anomary_like_normal.txt"

SPLIT_BASE_DIR = OUTPUT_DIR / "audio_dataset_split"
TRAIN_NORMAL_DIR = SPLIT_BASE_DIR / "train" / "normal"
EVAL_NORMAL_DIR = SPLIT_BASE_DIR / "eval" / "normal"
EVAL_ANOMARY_DIR = SPLIT_BASE_DIR / "eval" / "anomary"
AUDIO_CACHE_DIR = OUTPUT_DIR / "audio_cache"
EVALUATION_RESULTS_PATH = OUTPUT_DIR / "audio_evaluation_results.csv"
SPLIT_MANIFEST_PATH = SPLIT_BASE_DIR / "split_manifest.csv"

VIDEO_PROCESSING_ENABLED = True
VIDEO_SAMPLE_FPS = 2
VIDEO_MAX_SAMPLE_FRAMES = 200
VIDEO_LETTERBOX_BLACK_THRESHOLD = 8
VIDEO_UNIFORM_DOMINANT_RATIO_THRESHOLD = 0.95
VIDEO_UNIFORM_COLOR_TOLERANCE = 30.0
VEHICLE_SCORE_HIST_BINS = 64
VEHICLE_SCORE_QUANTILES = [0.25, 0.5, 0.75]
VEHICLE_LOW_SEGMENT_MAX = 1
VEHICLE_COMPONENT_MIN_AREA_RATIO = 0.002
VEHICLE_COMPONENT_KERNEL_SIZE = 5
EDGE_CANNY_LOW_THRESHOLD = 60
EDGE_CANNY_HIGH_THRESHOLD = 160
EDGE_PERSISTENCE_THRESHOLD = 0.4
EDGE_PREVIEW_FRAME_COUNT = 5
EDGE_PERSISTENCE_WEAK_THRESHOLD_RATIO = 0.6
EDGE_HYSTERESIS_KERNEL_SIZE = 5
EDGE_HYSTERESIS_MAX_ITERATIONS = 128
EDGE_ORIENTATION_TOLERANCE_DEG = 25.0
EDGE_COHERENCE_KERNEL_SIZE = 3
EDGE_COHERENCE_EPS = 1e-6
EDGE_COHERENCE_MIN_PERSISTENCE = 0.1
EDGE_COHERENCE_HIGH_QUANTILE = 0.8
EDGE_LOW_COHERENCE_QUANTILE = 0.2
EDGE_LOW_COHERENCE_REGION_RATIO_THRESHOLD = 0.4
EDGE_COHERENCE_MIN_COMPONENT_AREA_RATIO = 0.0005
EDGE_COMBINED_SCORE_TARGET_QUANTILE = 0.8
EDGE_DISPLAY_CONNECTED_MIN_PERSISTENCE_RATIO = 0.5
EDGE_DISPLAY_CONNECTED_COHERENCE_QUANTILE = 0.7
EDGE_DISPLAY_CONNECT_KERNEL_SIZE = 3
EDGE_DISPLAY_CONNECT_ITERATIONS = 1
EDGE_DISPLAY_BORDER_MAX_DISTANCE = 60.0
EDGE_DISPLAY_BORDER_DIRECTION_RADIUS = 12
EDGE_DISPLAY_BORDER_DIRECTION_NEIGHBORS = 6
EDGE_CONCAVE_HULL_ALPHA_RADIUS = 12.0
EDGE_CONCAVE_HULL_POINT_STEP = 2
EDGE_FINAL_CONNECT_KERNEL_SIZE = 9
EDGE_FINAL_CONNECT_ITERATIONS = 1
EDGE_FINAL_MIN_RECT_AREA_PIXELS = 518
EDGE_RAW_CONNECT_KERNEL_SIZE = 3
EDGE_RAW_CONNECT_ITERATIONS = 1
VEHICLE_GROWTH_SEED_SEGMENT_MAX = 0
VEHICLE_GROWTH_CANDIDATE_SEGMENT_MAX = 1
VEHICLE_GROWTH_BARRIER_KERNEL_SIZE = 3
VEHICLE_GROWTH_KERNEL_SIZE = 5
VEHICLE_GROWTH_REFINEMENT_KERNEL_SIZE = 5
VEHICLE_GROWTH_MAX_ITERATIONS = 128
VEHICLE_GROWTH_MIN_AREA_RATIO = 0.002

TRAIN_SPLIT_RATIO = 0.8
SPLIT_RANDOM_SEED = 42

AUDIO_SAMPLE_RATE = 22050
AUDIO_MONO_CHANNELS = 1
AUDIO_HIGHPASS_CUTOFF_HZ = 20.0
AUDIO_N_FFT = 1024
AUDIO_WIN_LENGTH = 1024
AUDIO_HOP_LENGTH = 256
AUDIO_N_MELS = 64
AUDIO_FMIN_HZ = 20.0
AUDIO_FMAX_HZ = 10000.0
AUDIO_WINDOW_SEC = 0.5
AUDIO_WINDOW_HOP_SEC = 0.25
AUDIO_BAND_LIMITS_HZ = [(20, 200), (200, 1000), (1000, 4000), (4000, 10000)]
AUDIO_PCA_COMPONENTS = 32
AUDIO_ISOLATION_FOREST_RANDOM_STATE = 42
AUDIO_CLIP_SCORE_TOP_K = 3
NORMAL_SCORE_THRESHOLD_PERCENTILE = 95.0


## 1. 動画の読み込み

In [ ]:
def resolve_reference_video_path() -> Path | None:
    if REFERENCE_VIDEO_PATH.exists():
        return REFERENCE_VIDEO_PATH

    normal_candidates = sorted(path for path in NORMAL_VIDEO_DIR.glob("*.mp4") if path.is_file())
    if normal_candidates:
        return normal_candidates[0]
    return None


def read_video_metadata(video_path: Path) -> dict[str, Any]:
    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        raise FileNotFoundError(f"Failed to open video: {video_path}")

    metadata = {
        "video_path": str(video_path),
        "frame_count": int(capture.get(cv2.CAP_PROP_FRAME_COUNT)),
        "fps": float(capture.get(cv2.CAP_PROP_FPS)),
        "width": int(capture.get(cv2.CAP_PROP_FRAME_WIDTH)),
        "height": int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT)),
    }
    metadata["duration_sec"] = (
        metadata["frame_count"] / metadata["fps"] if metadata["fps"] > 0 else np.nan
    )
    capture.release()
    return metadata


def iter_sampled_frames(
    video_path: Path,
    sampling_fps: int = VIDEO_SAMPLE_FPS,
    max_frames: int | None = VIDEO_MAX_SAMPLE_FRAMES,
) -> list[np.ndarray]:
    metadata = read_video_metadata(video_path)
    native_fps = metadata["fps"] if metadata["fps"] > 0 else 1.0
    step = max(int(round(native_fps / max(sampling_fps, 1))), 1)

    frames: list[np.ndarray] = []
    capture = cv2.VideoCapture(str(video_path))
    frame_index = 0
    while capture.isOpened():
        success, frame_bgr = capture.read()
        if not success:
            break

        if frame_index % step == 0:
            frames.append(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))
            if max_frames is not None and len(frames) >= max_frames:
                break
        frame_index += 1

    capture.release()
    return frames


def show_image(image: np.ndarray, title: str, figsize: tuple[int, int] = (8, 4)) -> None:
    figure, ax = plt.subplots(figsize=figsize)
    ax.imshow(image)
    ax.set_title(title)
    ax.axis("off")
    buffer = BytesIO()
    figure.savefig(buffer, format="png", bbox_inches="tight")
    plt.close(figure)
    buffer.seek(0)
    display(Image(data=buffer.getvalue()))


def show_video_figure(figure: plt.Figure) -> None:
    buffer = BytesIO()
    figure.savefig(buffer, format="png", bbox_inches="tight")
    plt.close(figure)
    buffer.seek(0)
    display(Image(data=buffer.getvalue()))


reference_video_path = resolve_reference_video_path()
reference_metadata = None
reference_sampled_stacked_frames: list[np.ndarray] = []
reference_stacked_frame = None

if reference_video_path is None:
    print("No reference video is available for vehicle-region segmentation.")
else:
    print("reference video:", reference_video_path)
    reference_metadata = read_video_metadata(reference_video_path)
    print("video metadata:")
    print(reference_metadata)

    reference_sampled_stacked_frames = iter_sampled_frames(reference_video_path)
    print("sampled stacked frame count:", len(reference_sampled_stacked_frames))

    if reference_sampled_stacked_frames:
        reference_stacked_frame = reference_sampled_stacked_frames[0]
        print("first stacked frame shape:", reference_stacked_frame.shape)
        show_image(reference_stacked_frame, title="Loaded First Stacked Frame")
    else:
        print("No sampled stacked frames could be loaded from the reference video.")


## 1.5 暫定: 左右レターボックス除去

In [ ]:
def detect_side_letterbox_bounds(
    stacked_frame: np.ndarray,
    black_threshold: int = VIDEO_LETTERBOX_BLACK_THRESHOLD,
    min_non_black_ratio: float = 0.01,
) -> tuple[int, int, dict[str, int]]:
    gray = cv2.cvtColor(stacked_frame, cv2.COLOR_RGB2GRAY)
    non_black_ratio = (gray > black_threshold).mean(axis=0)
    valid_columns = np.where(non_black_ratio > min_non_black_ratio)[0]

    if valid_columns.size == 0:
        crop_left = 0
        crop_right = stacked_frame.shape[1]
    else:
        crop_left = int(valid_columns[0])
        crop_right = int(valid_columns[-1]) + 1

    if crop_right <= crop_left:
        crop_left = 0
        crop_right = stacked_frame.shape[1]

    return crop_left, crop_right, {
        "original_width": int(stacked_frame.shape[1]),
        "crop_left": int(crop_left),
        "crop_right": int(crop_right),
        "cropped_width": int(crop_right - crop_left),
        "removed_left_px": int(crop_left),
        "removed_right_px": int(stacked_frame.shape[1] - crop_right),
    }


def apply_side_crop(stacked_frame: np.ndarray, crop_left: int, crop_right: int) -> np.ndarray:
    return stacked_frame[:, crop_left:crop_right, :]


def choose_common_side_letterbox_crop(stacked_frames: list[np.ndarray]) -> dict[str, int]:
    if not stacked_frames:
        return {
            "frame_count": 0,
            "original_width": 0,
            "crop_left": 0,
            "crop_right": 0,
            "cropped_width": 0,
            "removed_left_px": 0,
            "removed_right_px": 0,
        }

    crop_lefts: list[int] = []
    crop_rights: list[int] = []
    original_width = int(stacked_frames[0].shape[1])
    for stacked_frame in stacked_frames:
        crop_left, crop_right, _ = detect_side_letterbox_bounds(stacked_frame)
        crop_lefts.append(crop_left)
        crop_rights.append(crop_right)

    crop_left = int(np.median(crop_lefts))
    crop_right = int(np.median(crop_rights))
    crop_left = max(0, min(crop_left, original_width - 1))
    crop_right = max(crop_left + 1, min(crop_right, original_width))

    return {
        "frame_count": int(len(stacked_frames)),
        "original_width": original_width,
        "crop_left": int(crop_left),
        "crop_right": int(crop_right),
        "cropped_width": int(crop_right - crop_left),
        "removed_left_px": int(crop_left),
        "removed_right_px": int(original_width - crop_right),
    }


def preprocess_stacked_frames_remove_letterbox(
    stacked_frames: list[np.ndarray],
) -> tuple[list[np.ndarray], dict[str, int]]:
    if not stacked_frames:
        return [], choose_common_side_letterbox_crop(stacked_frames)

    crop_info = choose_common_side_letterbox_crop(stacked_frames)
    cropped_frames = [
        apply_side_crop(stacked_frame, crop_info["crop_left"], crop_info["crop_right"])
        for stacked_frame in stacked_frames
    ]
    return cropped_frames, crop_info


reference_preprocessed_stacked_frames: list[np.ndarray] = []
reference_preprocessed_frame = None
reference_letterbox_crop_info = None

if not reference_sampled_stacked_frames:
    print("Skip temporary letterbox removal because no sampled frames are available.")
else:
    reference_preprocessed_stacked_frames, reference_letterbox_crop_info = preprocess_stacked_frames_remove_letterbox(
        reference_sampled_stacked_frames
    )
    reference_preprocessed_frame = reference_preprocessed_stacked_frames[0]
    print("temporary common letterbox crop info:")
    print(reference_letterbox_crop_info)
    show_image(reference_stacked_frame, title="Before Temporary Letterbox Removal")
    show_image(reference_preprocessed_frame, title="After Temporary Letterbox Removal")


## 2. 前後動画分割

In [ ]:
def split_front_rear_frame(stacked_frame: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    height = stacked_frame.shape[0]
    midpoint = height // 2
    return stacked_frame[:midpoint, :, :], stacked_frame[midpoint:, :, :]


def split_front_rear_frames(stacked_frames: list[np.ndarray]) -> tuple[list[np.ndarray], list[np.ndarray]]:
    front_frames: list[np.ndarray] = []
    rear_frames: list[np.ndarray] = []
    for stacked_frame in stacked_frames:
        front_frame, rear_frame = split_front_rear_frame(stacked_frame)
        front_frames.append(front_frame)
        rear_frames.append(rear_frame)
    return front_frames, rear_frames


reference_front_frames: list[np.ndarray] = []
reference_rear_frames: list[np.ndarray] = []
reference_front_frame = None
reference_rear_frame = None

if not reference_preprocessed_stacked_frames:
    print("Skip split preview because no preprocessed frames are available.")
else:
    reference_front_frames, reference_rear_frames = split_front_rear_frames(reference_preprocessed_stacked_frames)
    print("split frame pairs:", len(reference_front_frames))
    if reference_front_frames and reference_rear_frames:
        reference_front_frame = reference_front_frames[0]
        reference_rear_frame = reference_rear_frames[0]
        print("front frame shape:", reference_front_frame.shape)
        print("rear frame shape:", reference_rear_frame.shape)
        show_image(reference_front_frame, title="Front First Frame Before Uniform-Frame Trim")
        show_image(reference_rear_frame, title="Rear First Frame Before Uniform-Frame Trim")


## 2.5 ほぼ一色のフレーム除去（前後同期）

In [ ]:
def is_nearly_uniform_frame(
    frame: np.ndarray,
    dominant_ratio_threshold: float = VIDEO_UNIFORM_DOMINANT_RATIO_THRESHOLD,
    color_tolerance: float = VIDEO_UNIFORM_COLOR_TOLERANCE,
) -> bool:
    pixels = frame.reshape(-1, 3).astype(np.float32)
    representative_color = np.median(pixels, axis=0)
    pixel_deviation = np.abs(pixels - representative_color).max(axis=1)
    near_representative_ratio = float((pixel_deviation <= color_tolerance).mean())
    return near_representative_ratio >= dominant_ratio_threshold


def count_edge_uniform_frames(frames: list[np.ndarray], reverse: bool = False) -> int:
    ordered_frames = list(reversed(frames)) if reverse else frames
    count = 0
    for frame in ordered_frames:
        if is_nearly_uniform_frame(frame):
            count += 1
        else:
            break
    return count


def trim_synchronized_uniform_frames(
    front_frames: list[np.ndarray],
    rear_frames: list[np.ndarray],
) -> tuple[list[np.ndarray], list[np.ndarray], dict[str, int]]:
    if len(front_frames) != len(rear_frames):
        raise ValueError("Front and rear frame counts must match before uniform-frame trimming.")

    total_frames = len(front_frames)
    if total_frames == 0:
        return [], [], {
            "input_frame_pairs": 0,
            "removed_from_start": 0,
            "removed_from_end": 0,
            "remaining_frame_pairs": 0,
        }

    front_uniform_start = count_edge_uniform_frames(front_frames, reverse=False)
    rear_uniform_start = count_edge_uniform_frames(rear_frames, reverse=False)
    front_uniform_end = count_edge_uniform_frames(front_frames, reverse=True)
    rear_uniform_end = count_edge_uniform_frames(rear_frames, reverse=True)

    removed_from_start = max(front_uniform_start, rear_uniform_start)
    removed_from_end = max(front_uniform_end, rear_uniform_end)

    if removed_from_start + removed_from_end >= total_frames:
        trimmed_front_frames = []
        trimmed_rear_frames = []
    else:
        end_index = total_frames - removed_from_end if removed_from_end > 0 else total_frames
        trimmed_front_frames = front_frames[removed_from_start:end_index]
        trimmed_rear_frames = rear_frames[removed_from_start:end_index]

    return trimmed_front_frames, trimmed_rear_frames, {
        "input_frame_pairs": int(total_frames),
        "front_uniform_frames_at_start": int(front_uniform_start),
        "rear_uniform_frames_at_start": int(rear_uniform_start),
        "front_uniform_frames_at_end": int(front_uniform_end),
        "rear_uniform_frames_at_end": int(rear_uniform_end),
        "removed_from_start": int(removed_from_start),
        "removed_from_end": int(removed_from_end),
        "remaining_frame_pairs": int(len(trimmed_front_frames)),
    }


reference_uniform_trim_info = None

if not reference_front_frames or not reference_rear_frames:
    print("Skip synchronized uniform-frame trim because split frames are not available.")
else:
    reference_front_frames, reference_rear_frames, reference_uniform_trim_info = trim_synchronized_uniform_frames(
        reference_front_frames,
        reference_rear_frames,
    )
    print("uniform frame trim info:")
    print(reference_uniform_trim_info)
    if reference_front_frames and reference_rear_frames:
        reference_front_frame = reference_front_frames[0]
        reference_rear_frame = reference_rear_frames[0]
        show_image(reference_front_frame, title="Front First Frame After Uniform-Frame Trim")
        show_image(reference_rear_frame, title="Rear First Frame After Uniform-Frame Trim")


## 4. 持続性エッジの確認


In [ ]:
def sample_indices_evenly(length: int, max_samples: int = EDGE_PREVIEW_FRAME_COUNT) -> list[int]:
    if length <= 0:
        return []
    if length <= max_samples:
        return list(range(length))
    sampled_indices = np.linspace(0, length - 1, num=max_samples, dtype=int)
    return [int(index) for index in np.unique(sampled_indices)]


def compute_structure_tensor_coherence(
    score_map: np.ndarray,
    kernel_size: int = EDGE_COHERENCE_KERNEL_SIZE,
    eps: float = EDGE_COHERENCE_EPS,
) -> np.ndarray:
    gradient_x = cv2.Sobel(score_map.astype(np.float32), ddepth=cv2.CV_32F, dx=1, dy=0, ksize=3)
    gradient_y = cv2.Sobel(score_map.astype(np.float32), ddepth=cv2.CV_32F, dx=0, dy=1, ksize=3)

    jxx = cv2.boxFilter(
        gradient_x * gradient_x,
        ddepth=-1,
        ksize=(kernel_size, kernel_size),
        normalize=False,
        borderType=cv2.BORDER_REFLECT,
    )
    jyy = cv2.boxFilter(
        gradient_y * gradient_y,
        ddepth=-1,
        ksize=(kernel_size, kernel_size),
        normalize=False,
        borderType=cv2.BORDER_REFLECT,
    )
    jxy = cv2.boxFilter(
        gradient_x * gradient_y,
        ddepth=-1,
        ksize=(kernel_size, kernel_size),
        normalize=False,
        borderType=cv2.BORDER_REFLECT,
    )

    trace = jxx + jyy
    delta = np.sqrt(np.maximum((jxx - jyy) ** 2 + 4.0 * (jxy ** 2), 0.0))
    lambda_1 = 0.5 * (trace + delta)
    lambda_2 = 0.5 * (trace - delta)
    coherence_map = (lambda_1 - lambda_2) / (lambda_1 + lambda_2 + float(eps))
    return np.clip(coherence_map, 0.0, 1.0).astype(np.float32)


def select_valley_thresholds(
    histogram_counts: np.ndarray,
    histogram_edges: np.ndarray,
    target_thresholds: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    histogram_counts = np.asarray(histogram_counts, dtype=np.float32)
    histogram_edges = np.asarray(histogram_edges, dtype=np.float32)
    target_thresholds = np.asarray(target_thresholds, dtype=np.float32)

    if histogram_counts.size == 0:
        return target_thresholds.copy(), np.array([], dtype=np.int32), histogram_counts.copy()

    smooth_kernel = np.array([1.0, 2.0, 3.0, 2.0, 1.0], dtype=np.float32)
    smooth_kernel /= smooth_kernel.sum()
    smoothed_histogram_counts = np.convolve(histogram_counts, smooth_kernel, mode="same")

    valley_mask = np.zeros_like(smoothed_histogram_counts, dtype=bool)
    if smoothed_histogram_counts.size >= 3:
        valley_mask[1:-1] = (
            (smoothed_histogram_counts[1:-1] <= smoothed_histogram_counts[:-2])
            & (smoothed_histogram_counts[1:-1] <= smoothed_histogram_counts[2:])
        )
    valley_indices = np.where(valley_mask)[0].astype(np.int32)

    bin_centers = 0.5 * (histogram_edges[:-1] + histogram_edges[1:])
    selected_thresholds = []
    for target_threshold in target_thresholds:
        if valley_indices.size > 0:
            nearest_valley_index = valley_indices[np.argmin(np.abs(bin_centers[valley_indices] - float(target_threshold)))]
            selected_thresholds.append(float(bin_centers[nearest_valley_index]))
        else:
            nearest_bin_index = int(np.argmin(np.abs(bin_centers - float(target_threshold))))
            selected_thresholds.append(float(bin_centers[nearest_bin_index]))

    return np.asarray(selected_thresholds, dtype=np.float32), valley_indices, smoothed_histogram_counts.astype(np.float32)


def build_score_histogram(
    score_map: np.ndarray,
    score_range: tuple[float, float],
    target_threshold: float,
) -> dict[str, Any]:
    histogram_counts, histogram_edges = np.histogram(
        score_map.reshape(-1).astype(np.float32),
        bins=VEHICLE_SCORE_HIST_BINS,
        range=score_range,
    )
    selected_thresholds, valley_indices, smoothed_histogram_counts = select_valley_thresholds(
        histogram_counts=histogram_counts,
        histogram_edges=histogram_edges,
        target_thresholds=np.array([target_threshold], dtype=np.float32),
    )
    selected_threshold = float(selected_thresholds[0]) if selected_thresholds.size else float(target_threshold)
    return {
        "histogram_counts": histogram_counts,
        "histogram_edges": histogram_edges,
        "smoothed_histogram_counts": smoothed_histogram_counts,
        "valley_indices": valley_indices.astype(np.int32),
        "target_threshold": float(target_threshold),
        "selected_threshold": float(selected_threshold),
    }


def connect_small_edge_gaps(
    edge_mask: np.ndarray,
    kernel_size: int = EDGE_RAW_CONNECT_KERNEL_SIZE,
    iterations: int = EDGE_RAW_CONNECT_ITERATIONS,
) -> np.ndarray:
    kernel = np.ones((kernel_size, kernel_size), dtype=np.uint8)
    connected_mask = cv2.morphologyEx(
        edge_mask.astype(np.uint8),
        cv2.MORPH_CLOSE,
        kernel,
        iterations=iterations,
    )
    return connected_mask > 0


def filter_small_edge_components(
    edge_mask: np.ndarray,
    min_area_ratio: float = EDGE_COHERENCE_MIN_COMPONENT_AREA_RATIO,
) -> dict[str, Any]:
    edge_uint8 = edge_mask.astype(np.uint8)
    component_count, component_labels, component_stats, _ = cv2.connectedComponentsWithStats(edge_uint8, connectivity=8)
    minimum_area_pixels = max(1, int(round(min_area_ratio * edge_uint8.shape[0] * edge_uint8.shape[1])))

    filtered_mask = np.zeros_like(edge_uint8, dtype=np.uint8)
    kept_component_count = 0
    removed_component_count = 0
    for component_id in range(1, int(component_count)):
        area_pixels = int(component_stats[component_id, cv2.CC_STAT_AREA])
        if area_pixels < minimum_area_pixels:
            removed_component_count += 1
            continue
        filtered_mask[component_labels == component_id] = 1
        kept_component_count += 1

    return {
        "filtered_mask": filtered_mask > 0,
        "minimum_area_pixels": int(minimum_area_pixels),
        "kept_component_count": int(kept_component_count),
        "removed_component_count": int(removed_component_count),
    }


def filter_edge_components_by_min_rect_area(
    edge_mask: np.ndarray,
    min_rect_area_pixels: float = EDGE_FINAL_MIN_RECT_AREA_PIXELS,
) -> dict[str, Any]:
    edge_uint8 = edge_mask.astype(np.uint8)
    component_count, component_labels, _, _ = cv2.connectedComponentsWithStats(edge_uint8, connectivity=8)
    minimum_rect_area_pixels = max(1.0, float(min_rect_area_pixels))

    filtered_mask = np.zeros_like(edge_uint8, dtype=np.uint8)
    kept_component_count = 0
    removed_component_count = 0
    for component_id in range(1, int(component_count)):
        component_mask = component_labels == component_id
        point_y, point_x = np.where(component_mask)
        if point_x.size == 0:
            continue
        points = np.column_stack([point_x, point_y]).astype(np.float32)
        rect = cv2.minAreaRect(points)
        rect_area_pixels = float(rect[1][0] * rect[1][1])
        if rect_area_pixels < minimum_rect_area_pixels:
            removed_component_count += 1
            continue
        filtered_mask[component_mask] = 1
        kept_component_count += 1

    return {
        "filtered_mask": filtered_mask > 0,
        "minimum_rect_area_pixels": float(minimum_rect_area_pixels),
        "kept_component_count": int(kept_component_count),
        "removed_component_count": int(removed_component_count),
    }


def fill_enclosed_low_coherence_regions(
    edge_mask: np.ndarray,
    coherence_map: np.ndarray,
    low_coherence_quantile: float = EDGE_LOW_COHERENCE_QUANTILE,
    low_coherence_region_ratio_threshold: float = EDGE_LOW_COHERENCE_REGION_RATIO_THRESHOLD,
) -> dict[str, Any]:
    low_coherence_threshold = float(np.quantile(coherence_map.reshape(-1), low_coherence_quantile))
    low_coherence_mask = coherence_map <= low_coherence_threshold

    free_space_mask = (~edge_mask).astype(np.uint8)

    component_count, component_labels = cv2.connectedComponents(free_space_mask, connectivity=8)
    filled_region_mask = np.zeros_like(free_space_mask, dtype=np.uint8)
    candidate_region_count = 0
    filled_region_count = 0

    for component_id in range(1, int(component_count)):
        region_mask = component_labels == component_id
        if not np.any(region_mask):
            continue

        candidate_region_count += 1
        low_coherence_ratio = float(low_coherence_mask[region_mask].mean())
        if low_coherence_ratio >= float(low_coherence_region_ratio_threshold):
            filled_region_mask[region_mask] = 1
            filled_region_count += 1

    return {
        "low_coherence_mask": low_coherence_mask,
        "low_coherence_threshold": float(low_coherence_threshold),
        "filled_region_mask": filled_region_mask > 0,
        "candidate_region_count": int(candidate_region_count),
        "enclosed_region_count": int(candidate_region_count),
        "filled_region_count": int(filled_region_count),
        "filled_region_ratio": float((filled_region_mask > 0).mean()),
        "low_coherence_region_ratio_threshold": float(low_coherence_region_ratio_threshold),
    }


def build_edge_separated_region_image(edge_mask: np.ndarray) -> dict[str, Any]:
    free_space_mask = (~edge_mask).astype(np.uint8)
    component_count, component_labels, _, component_centroids = cv2.connectedComponentsWithStats(
        free_space_mask,
        connectivity=8,
    )
    region_count = max(0, int(component_count) - 1)

    region_label_image = np.zeros((*edge_mask.shape, 3), dtype=np.uint8)
    if region_count > 0:
        region_colors = (
            plt.cm.hsv(np.linspace(0.0, 1.0, num=region_count, endpoint=False))[:, :3] * 255.0
        ).astype(np.uint8)
        text_scale = max(0.3, min(edge_mask.shape) / 420.0)
        text_thickness = max(1, int(round(text_scale * 1.5)))
        for component_id in range(1, int(component_count)):
            region_label_image[component_labels == component_id] = region_colors[component_id - 1]
        region_label_image[edge_mask] = np.array([255, 255, 255], dtype=np.uint8)
        for component_id in range(1, int(component_count)):
            centroid_x, centroid_y = component_centroids[component_id]
            label_text = str(component_id)
            text_size, baseline = cv2.getTextSize(
                label_text,
                cv2.FONT_HERSHEY_SIMPLEX,
                text_scale,
                text_thickness,
            )
            text_origin = (
                int(round(centroid_x - text_size[0] / 2.0)),
                int(round(centroid_y + text_size[1] / 2.0)),
            )
            cv2.putText(
                region_label_image,
                label_text,
                text_origin,
                cv2.FONT_HERSHEY_SIMPLEX,
                text_scale,
                (0, 0, 0),
                thickness=text_thickness + 1,
                lineType=cv2.LINE_AA,
            )
            cv2.putText(
                region_label_image,
                label_text,
                text_origin,
                cv2.FONT_HERSHEY_SIMPLEX,
                text_scale,
                (255, 255, 255),
                thickness=text_thickness,
                lineType=cv2.LINE_AA,
            )

    return {
        "region_label_image": region_label_image,
        "region_count": int(region_count),
    }


def compute_persistent_edge_score(
    frames: list[np.ndarray],
    low_threshold: int = EDGE_CANNY_LOW_THRESHOLD,
    high_threshold: int = EDGE_CANNY_HIGH_THRESHOLD,
    persistence_threshold: float = EDGE_PERSISTENCE_THRESHOLD,
    coherence_high_quantile: float = EDGE_COHERENCE_HIGH_QUANTILE,
    coherence_min_persistence: float = EDGE_COHERENCE_MIN_PERSISTENCE,
    coherence_min_component_area_ratio: float = EDGE_COHERENCE_MIN_COMPONENT_AREA_RATIO,
    low_coherence_quantile: float = EDGE_LOW_COHERENCE_QUANTILE,
    low_coherence_region_ratio_threshold: float = EDGE_LOW_COHERENCE_REGION_RATIO_THRESHOLD,
) -> dict[str, Any]:
    if not frames:
        raise ValueError("At least one frame is required to compute persistent edges.")

    gray_frames = [
        cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY).astype(np.uint8)
        for frame in frames
    ]

    edge_masks = []
    for gray_frame in gray_frames:
        blurred = cv2.GaussianBlur(gray_frame, (5, 5), 0)
        edge_mask = cv2.Canny(blurred, threshold1=low_threshold, threshold2=high_threshold) > 0
        edge_masks.append(edge_mask)

    edge_stack = np.stack(edge_masks, axis=0)
    persistence_score = edge_stack.mean(axis=0).astype(np.float32)

    persistence_histogram = build_score_histogram(
        score_map=persistence_score,
        score_range=(0.0, 1.0),
        target_threshold=float(persistence_threshold),
    )
    strong_threshold = float(persistence_histogram["selected_threshold"])
    strong_edge_seed_mask = persistence_score >= strong_threshold

    high_coherence_input_score = persistence_score.copy()
    high_coherence_input_score[high_coherence_input_score <= float(coherence_min_persistence)] = 0.0
    high_coherence_map = compute_structure_tensor_coherence(high_coherence_input_score)
    low_coherence_input_score = persistence_score.copy()
    low_coherence_map = compute_structure_tensor_coherence(low_coherence_input_score)
    high_coherence_threshold = float(np.quantile(high_coherence_map.reshape(-1), coherence_high_quantile))
    raw_high_coherence_mask = high_coherence_map >= high_coherence_threshold
    high_coherence_component_filter_result = filter_small_edge_components(
        raw_high_coherence_mask,
        min_area_ratio=coherence_min_component_area_ratio,
    )
    high_coherence_mask = high_coherence_component_filter_result["filtered_mask"]

    weak_edge_mask = (persistence_score > 0.0) & (~strong_edge_seed_mask)
    high_coherence_weak_candidate_mask = weak_edge_mask & high_coherence_mask
    coherence_supported_weak_mask = high_coherence_weak_candidate_mask

    raw_persistent_edge_mask = (strong_edge_seed_mask | coherence_supported_weak_mask).astype(bool)
    gap_filled_persistent_edge_mask = connect_small_edge_gaps(
        raw_persistent_edge_mask,
        kernel_size=EDGE_RAW_CONNECT_KERNEL_SIZE,
        iterations=EDGE_RAW_CONNECT_ITERATIONS,
    )
    component_filter_result = filter_edge_components_by_min_rect_area(gap_filled_persistent_edge_mask)
    rect_filtered_persistent_edge_mask = component_filter_result["filtered_mask"]
    final_connected_persistent_edge_mask = connect_small_edge_gaps(
        rect_filtered_persistent_edge_mask,
        kernel_size=EDGE_FINAL_CONNECT_KERNEL_SIZE,
        iterations=EDGE_FINAL_CONNECT_ITERATIONS,
    )
    persistent_edge_mask = final_connected_persistent_edge_mask
    low_coherence_region_result = fill_enclosed_low_coherence_regions(
        edge_mask=persistent_edge_mask,
        coherence_map=low_coherence_map,
        low_coherence_quantile=low_coherence_quantile,
        low_coherence_region_ratio_threshold=low_coherence_region_ratio_threshold,
    )
    vehicle_mask = persistent_edge_mask | low_coherence_region_result["filled_region_mask"]

    return {
        "edge_stack": edge_stack,
        "persistence_score": persistence_score,
        "coherence_input_score": low_coherence_input_score,
        "coherence_map": low_coherence_map,
        "high_coherence_input_score": high_coherence_input_score,
        "high_coherence_map": high_coherence_map,
        "strong_edge_seed_mask": strong_edge_seed_mask,
        "weak_edge_mask": weak_edge_mask,
        "raw_high_coherence_mask": raw_high_coherence_mask,
        "high_coherence_mask": high_coherence_mask,
        "high_coherence_weak_candidate_mask": high_coherence_weak_candidate_mask,
        "coherence_supported_weak_mask": coherence_supported_weak_mask,
        "raw_persistent_edge_mask": raw_persistent_edge_mask,
        "gap_filled_persistent_edge_mask": gap_filled_persistent_edge_mask,
        "rect_filtered_persistent_edge_mask": rect_filtered_persistent_edge_mask,
        "final_connected_persistent_edge_mask": final_connected_persistent_edge_mask,
        "persistent_edge_mask": persistent_edge_mask,
        "low_coherence_mask": low_coherence_region_result["low_coherence_mask"],
        "filled_low_coherence_region_mask": low_coherence_region_result["filled_region_mask"],
        "vehicle_mask": vehicle_mask,
        "frame_count": int(len(frames)),
        "edge_pixel_ratio_mean": float(edge_stack.mean()),
        "strong_edge_seed_ratio": float(strong_edge_seed_mask.mean()),
        "weak_edge_ratio": float(weak_edge_mask.mean()),
        "high_coherence_ratio": float(high_coherence_mask.mean()),
        "coherence_supported_weak_ratio": float(coherence_supported_weak_mask.mean()),
        "raw_persistent_edge_ratio": float(raw_persistent_edge_mask.mean()),
        "gap_filled_persistent_edge_ratio": float(gap_filled_persistent_edge_mask.mean()),
        "rect_filtered_persistent_edge_ratio": float(rect_filtered_persistent_edge_mask.mean()),
        "final_connected_persistent_edge_ratio": float(final_connected_persistent_edge_mask.mean()),
        "persistent_edge_ratio": float(persistent_edge_mask.mean()),
        "vehicle_mask_ratio": float(vehicle_mask.mean()),
        "minimum_component_rect_area_pixels": component_filter_result["minimum_rect_area_pixels"],
        "kept_component_count": component_filter_result["kept_component_count"],
        "removed_component_count": component_filter_result["removed_component_count"],
        "coherence_mean": float(low_coherence_map.mean()),
        "high_coherence_map_mean": float(high_coherence_map.mean()),
        "coherence_min_persistence": float(coherence_min_persistence),
        "coherence_min_component_area_pixels": high_coherence_component_filter_result["minimum_area_pixels"],
        "removed_high_coherence_component_count": high_coherence_component_filter_result["removed_component_count"],
        "high_coherence_threshold": float(high_coherence_threshold),
        "coherence_high_quantile": float(coherence_high_quantile),
        "low_coherence_threshold": low_coherence_region_result["low_coherence_threshold"],
        "low_coherence_quantile": float(low_coherence_quantile),
        "low_coherence_region_ratio_threshold": low_coherence_region_result["low_coherence_region_ratio_threshold"],
        "candidate_region_count": low_coherence_region_result["candidate_region_count"],
        "enclosed_region_count": low_coherence_region_result["enclosed_region_count"],
        "filled_region_count": low_coherence_region_result["filled_region_count"],
        "filled_region_ratio": low_coherence_region_result["filled_region_ratio"],
        "persistence_threshold": float(strong_threshold),
        "target_persistence_threshold": float(persistence_histogram["target_threshold"]),
        "persistence_histogram_counts": persistence_histogram["histogram_counts"],
        "persistence_histogram_edges": persistence_histogram["histogram_edges"],
        "persistence_smoothed_histogram_counts": persistence_histogram["smoothed_histogram_counts"],
        "persistence_valley_indices": persistence_histogram["valley_indices"],
    }


def build_persistent_edge_figure(
    view_name: str,
    reference_frame: np.ndarray,
    edge_result: dict[str, Any],
) -> plt.Figure:
    figure, axes = plt.subplots(3, 3, figsize=(28, 16), constrained_layout=True)
    axes = axes.reshape(3, 3)

    persistence_score = edge_result["persistence_score"]
    coherence_map = edge_result["coherence_map"]
    low_coherence_mask = edge_result["low_coherence_mask"]
    strong_edge_seed_mask = edge_result["strong_edge_seed_mask"]
    gap_filled_persistent_edge_mask = edge_result["gap_filled_persistent_edge_mask"]
    persistent_edge_mask = edge_result["persistent_edge_mask"]
    filled_low_coherence_region_mask = edge_result["filled_low_coherence_region_mask"]
    separated_region_result = build_edge_separated_region_image(persistent_edge_mask)
    separated_region_label_image = separated_region_result["region_label_image"]
    separated_region_count = separated_region_result["region_count"]

    persistence_image = axes[0, 0].imshow(persistence_score, cmap="magma", vmin=0.0, vmax=1.0)
    axes[0, 0].set_title(f"{view_name} edge persistence score")
    axes[0, 0].axis("off")
    figure.colorbar(persistence_image, ax=axes[0, 0], fraction=0.046, pad=0.04)

    coherence_image = axes[0, 1].imshow(coherence_map, cmap="viridis", vmin=0.0, vmax=1.0)
    axes[0, 1].set_title(f"{view_name} coherence map\n(raw persistence)")
    axes[0, 1].axis("off")
    figure.colorbar(coherence_image, ax=axes[0, 1], fraction=0.046, pad=0.04)

    axes[0, 2].imshow(strong_edge_seed_mask, cmap="gray", vmin=0.0, vmax=1.0)
    axes[0, 2].set_title(f"{view_name} strong edge seeds")
    axes[0, 2].axis("off")

    axes[1, 0].imshow(low_coherence_mask, cmap="gray", vmin=0.0, vmax=1.0)
    axes[1, 0].set_title(f"{view_name} low-coherence mask")
    axes[1, 0].axis("off")

    axes[1, 1].imshow(gap_filled_persistent_edge_mask, cmap="gray", vmin=0.0, vmax=1.0)
    axes[1, 1].set_title(f"{view_name} strong edge + high-coherence weak mask\n(gap-filled)")
    axes[1, 1].axis("off")

    axes[1, 2].imshow(persistent_edge_mask, cmap="gray", vmin=0.0, vmax=1.0)
    axes[1, 2].set_title(f"{view_name} final persistent edge mask")
    axes[1, 2].axis("off")

    axes[2, 0].imshow(filled_low_coherence_region_mask, cmap="gray", vmin=0.0, vmax=1.0)
    axes[2, 0].set_title(f"{view_name} low-coherence-filled regions")
    axes[2, 0].axis("off")

    axes[2, 1].imshow(separated_region_label_image)
    axes[2, 1].set_title(f"{view_name} edge-separated regions\n(count={separated_region_count})")
    axes[2, 1].axis("off")

    final_mask_overlay = np.repeat((persistent_edge_mask.astype(np.uint8) * 255)[:, :, None], 3, axis=2)
    low_coherence_only_mask = low_coherence_mask & (~persistent_edge_mask)
    low_coherence_on_edge_mask = low_coherence_mask & persistent_edge_mask
    final_mask_overlay[low_coherence_only_mask] = np.array([0, 255, 255], dtype=np.uint8)
    final_mask_overlay[low_coherence_on_edge_mask] = np.array([255, 255, 0], dtype=np.uint8)
    axes[2, 2].imshow(final_mask_overlay)
    axes[2, 2].set_title(f"{view_name} final persistent edge + low-coherence overlay")
    axes[2, 2].axis("off")

    return figure

persistent_edge_results: dict[str, Any] = {}

if not reference_front_frames or not reference_rear_frames:
    print("Skip persistent-edge analysis because aligned front/rear frames are not available.")
else:
    for view_name, frames in {
        "front": reference_front_frames,
        "rear": reference_rear_frames,
    }.items():
        edge_result = compute_persistent_edge_score(frames)
        persistent_edge_results[view_name] = {
            "edge_result": edge_result,
            "reference_frame": frames[0],
            "vehicle_mask": edge_result["vehicle_mask"],
        }

        print("-" * 80)
        print("view:", view_name)
        print("frame count:", edge_result["frame_count"])
        print("mean edge pixel ratio:", edge_result["edge_pixel_ratio_mean"])
        print("target persistent edge threshold:", edge_result["target_persistence_threshold"])
        print("strong persistence threshold:", edge_result["persistence_threshold"])
        print("persistence valley candidate count:", len(edge_result["persistence_valley_indices"]))
        print("strong edge seed ratio:", edge_result["strong_edge_seed_ratio"])
        print("weak edge ratio:", edge_result["weak_edge_ratio"])
        print("high-coherence ratio:", edge_result["high_coherence_ratio"])
        print("high-coherence weak candidate ratio:", float(edge_result["high_coherence_weak_candidate_mask"].mean()))
        print("high-coherence weak ratio:", edge_result["coherence_supported_weak_ratio"])
        print("raw persistent edge ratio:", edge_result["raw_persistent_edge_ratio"])
        print("gap-filled persistent edge ratio:", edge_result["gap_filled_persistent_edge_ratio"])
        print("rect-filtered persistent edge ratio:", edge_result["rect_filtered_persistent_edge_ratio"])
        print("final-connected persistent edge ratio:", edge_result["final_connected_persistent_edge_ratio"])
        print("persistent edge ratio:", edge_result["persistent_edge_ratio"])
        print("vehicle mask ratio:", edge_result["vehicle_mask_ratio"])
        print("minimum component rect area pixels:", edge_result["minimum_component_rect_area_pixels"])
        print("kept component count:", edge_result["kept_component_count"])
        print("removed small component count:", edge_result["removed_component_count"])
        print("coherence min persistence:", edge_result["coherence_min_persistence"])
        print("coherence min component area pixels:", edge_result["coherence_min_component_area_pixels"])
        print("removed high-coherence component count:", edge_result["removed_high_coherence_component_count"])
        print("high coherence threshold:", edge_result["high_coherence_threshold"])
        print("low coherence threshold:", edge_result["low_coherence_threshold"])
        print("coherence mean:", edge_result["coherence_mean"])
        print("high-coherence map mean:", edge_result["high_coherence_map_mean"])
        print("candidate region count:", edge_result["candidate_region_count"])
        print("filled region count:", edge_result["filled_region_count"])
        print("filled region ratio:", edge_result["filled_region_ratio"])
        show_video_figure(
            build_persistent_edge_figure(
                view_name=view_name,
                reference_frame=frames[0],
                edge_result=edge_result,
            )
        )


## 5. 床検出前処理プレビュー（2.5処理後フレーム）

In [ ]:
def summarize_masked_values(values: np.ndarray, mask: np.ndarray, prefix: str) -> dict[str, float]:
    masked_values = values[mask]
    if masked_values.size == 0:
        return {
            f"{prefix}_pixel_count": 0.0,
            f"{prefix}_mean": float("nan"),
            f"{prefix}_std": float("nan"),
        }
    return {
        f"{prefix}_pixel_count": float(masked_values.size),
        f"{prefix}_mean": float(masked_values.mean()),
        f"{prefix}_std": float(masked_values.std()),
    }


def mask_preview_image(frame_rgb: np.ndarray, processing_mask: np.ndarray) -> np.ndarray:
    masked_frame = np.zeros_like(frame_rgb, dtype=np.uint8)
    masked_frame[processing_mask] = frame_rgb[processing_mask]
    return masked_frame


def build_hs_only_image(
    frame_rgb: np.ndarray,
    processing_mask: np.ndarray,
    fixed_value: int = 255,
) -> tuple[np.ndarray, dict[str, float]]:
    frame_hsv = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2HSV)
    hue_channel = frame_hsv[:, :, 0]
    saturation_channel = frame_hsv[:, :, 1]
    value_channel = frame_hsv[:, :, 2]

    fixed_value_uint8 = np.uint8(np.clip(fixed_value, 0, 255))

    hs_only_hsv = np.dstack([
        hue_channel,
        saturation_channel,
        np.full_like(value_channel, fixed_value_uint8),
    ])
    hs_only_rgb = cv2.cvtColor(hs_only_hsv, cv2.COLOR_HSV2RGB)
    masked_hs_only_rgb = mask_preview_image(hs_only_rgb, processing_mask)
    hsv_stats = {
        "processing_pixel_count": float(processing_mask.sum()),
        "hs_only_value_channel": float(fixed_value_uint8),
    }
    hsv_stats.update(summarize_masked_values(hue_channel, processing_mask, "hue"))
    hsv_stats.update(summarize_masked_values(saturation_channel, processing_mask, "saturation"))
    hsv_stats.update(summarize_masked_values(value_channel, processing_mask, "value"))
    return masked_hs_only_rgb, hsv_stats


def apply_clahe_luminance_equalization(
    frame_rgb: np.ndarray,
    processing_mask: np.ndarray,
    clip_limit: float = 2.0,
    tile_grid_size: tuple[int, int] = (8, 8),
) -> tuple[np.ndarray, dict[str, Any]]:
    frame_lab = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2LAB)
    lightness_channel, a_channel, b_channel = cv2.split(frame_lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    equalized_lightness = clahe.apply(lightness_channel)
    equalized_lab = cv2.merge([equalized_lightness, a_channel, b_channel])
    equalized_rgb = cv2.cvtColor(equalized_lab, cv2.COLOR_LAB2RGB)
    masked_equalized_rgb = mask_preview_image(equalized_rgb, processing_mask)
    clahe_stats = {
        "clip_limit": float(clip_limit),
        "tile_grid_size": tuple(int(value) for value in tile_grid_size),
        "processing_pixel_count": float(processing_mask.sum()),
    }
    clahe_stats.update(summarize_masked_values(lightness_channel, processing_mask, "lightness_before"))
    clahe_stats.update(summarize_masked_values(equalized_lightness, processing_mask, "lightness_after"))
    return masked_equalized_rgb, clahe_stats


def apply_single_scale_retinex(
    frame_rgb: np.ndarray,
    processing_mask: np.ndarray,
    sigma: float = 80.0,
) -> tuple[np.ndarray, dict[str, Any]]:
    frame_float = frame_rgb.astype(np.float32) + 1.0
    retinex_channels = []
    for channel_index in range(frame_float.shape[2]):
        channel = frame_float[:, :, channel_index]
        blurred_channel = cv2.GaussianBlur(channel, (0, 0), sigmaX=sigma, sigmaY=sigma)
        retinex_channel = np.log(channel) - np.log(blurred_channel + 1.0)
        normalized_channel = cv2.normalize(retinex_channel, None, alpha=0, beta=255, norm_type=cv2.NORM_MINMAX)
        retinex_channels.append(normalized_channel.astype(np.uint8))
    retinex_rgb = np.stack(retinex_channels, axis=2)
    masked_retinex_rgb = mask_preview_image(retinex_rgb, processing_mask)
    intensity_before = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2GRAY)
    intensity_after = cv2.cvtColor(retinex_rgb, cv2.COLOR_RGB2GRAY)
    retinex_stats = {
        "retinex_sigma": float(sigma),
        "processing_pixel_count": float(processing_mask.sum()),
    }
    retinex_stats.update(summarize_masked_values(intensity_before, processing_mask, "retinex_intensity_before"))
    retinex_stats.update(summarize_masked_values(intensity_after, processing_mask, "retinex_intensity_after"))
    return masked_retinex_rgb, retinex_stats


floor_preview_results: dict[str, dict[str, Any]] = {}
floor_preview_source_results = persistent_edge_results if "persistent_edge_results" in globals() else {}

if not floor_preview_source_results:
    print("Skip floor-detection preprocessing preview because section 4 vehicle masks are not available.")
else:
    print("Use the first front/rear frames after section 2.5 synchronized uniform-frame trimming.")
    print("Section 5 processes only non-vehicle regions using the vehicle mask from section 4.")
    print("HS-only view fixes V=255, and Retinex HS-only view fixes V=128 for comparison.")
    for view_name in ["front", "rear"]:
        if view_name not in floor_preview_source_results:
            continue
        floor_preview_source_result = floor_preview_source_results[view_name]
        floor_preview_frame = floor_preview_source_result.get("reference_frame")
        floor_preview_vehicle_mask = floor_preview_source_result.get("vehicle_mask")
        if floor_preview_vehicle_mask is None and "edge_result" in floor_preview_source_result:
            floor_preview_edge_result = floor_preview_source_result["edge_result"]
            floor_preview_vehicle_mask = (
                floor_preview_edge_result["persistent_edge_mask"]
                | floor_preview_edge_result["filled_low_coherence_region_mask"]
            )
        if floor_preview_frame is None or floor_preview_vehicle_mask is None:
            print("Skip floor preview for view because frame or vehicle mask is missing:", view_name)
            continue

        floor_preview_vehicle_mask = np.asarray(floor_preview_vehicle_mask, dtype=bool)
        if floor_preview_vehicle_mask.shape != floor_preview_frame.shape[:2]:
            print("Skip floor preview for view because frame/mask shapes do not match:", view_name)
            continue

        floor_preview_processing_mask = ~floor_preview_vehicle_mask
        floor_preview_hs_only_frame, floor_preview_hsv_stats = build_hs_only_image(
            floor_preview_frame,
            floor_preview_processing_mask,
        )
        floor_preview_clahe_frame, floor_preview_clahe_stats = apply_clahe_luminance_equalization(
            floor_preview_frame,
            floor_preview_processing_mask,
        )
        floor_preview_retinex_frame, floor_preview_retinex_stats = apply_single_scale_retinex(
            floor_preview_frame,
            floor_preview_processing_mask,
        )
        floor_preview_retinex_hs_only_frame, floor_preview_retinex_hs_only_stats = build_hs_only_image(
            floor_preview_retinex_frame,
            floor_preview_processing_mask,
            fixed_value=128,
        )
        floor_preview_results[view_name] = {
            "frame": floor_preview_frame,
            "vehicle_mask": floor_preview_vehicle_mask,
            "processing_mask": floor_preview_processing_mask,
            "vehicle_mask_ratio": float(floor_preview_vehicle_mask.mean()),
            "processing_mask_ratio": float(floor_preview_processing_mask.mean()),
            "hs_only_frame": floor_preview_hs_only_frame,
            "clahe_frame": floor_preview_clahe_frame,
            "retinex_frame": floor_preview_retinex_frame,
            "retinex_hs_only_frame": floor_preview_retinex_hs_only_frame,
            "hsv_stats": floor_preview_hsv_stats,
            "clahe_stats": floor_preview_clahe_stats,
            "retinex_stats": floor_preview_retinex_stats,
            "retinex_hs_only_stats": floor_preview_retinex_hs_only_stats,
        }

        print("-" * 80)
        print("floor preview view:", view_name)
        print("floor preview frame source:", f"reference_{view_name}_frame_after_section_2_5")
        print("floor preview frame shape:", floor_preview_frame.shape)
        print("vehicle mask ratio:", float(floor_preview_vehicle_mask.mean()))
        print("non-vehicle processing ratio:", float(floor_preview_processing_mask.mean()))
        print("HSV channel stats (non-vehicle region only):")
        print(floor_preview_hsv_stats)
        print("CLAHE lightness stats (non-vehicle region only):")
        print(floor_preview_clahe_stats)
        print("Retinex intensity stats (non-vehicle region only):")
        print(floor_preview_retinex_stats)
        print("Retinex HS-only stats (non-vehicle region only, V=128):")
        print(floor_preview_retinex_hs_only_stats)
        show_image(
            floor_preview_hs_only_frame,
            title=f"Floor Detection Preview ({view_name}): HSV HS-Only Image (non-vehicle region)",
        )
        show_image(
            floor_preview_clahe_frame,
            title=f"Floor Detection Preview ({view_name}): CLAHE Equalized Image (non-vehicle region)",
        )
        show_image(
            floor_preview_retinex_frame,
            title=f"Floor Detection Preview ({view_name}): Retinex Image (non-vehicle region)",
        )
        show_image(
            floor_preview_retinex_hs_only_frame,
            title=f"Floor Detection Preview ({view_name}): Retinex HS-Only Image (V=128, non-vehicle region)",
        )
